# Day 51: Production Dockerfile for vLLM

**Layer:** Production


## Concept Overview

A production Dockerfile packages the model runtime, startup script, health checks, and metrics endpoint. Key practices: multi-stage build to minimize image size, non-root user, entrypoint that waits for model load before accepting traffic.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Production Dockerfile

A production Dockerfile packages the model runtime, startup script, health checks, and metrics endpo...


In [ ]:
print("Production vLLM Dockerfile:")
print("""
FROM nvcr.io/nvidia/cuda:12.4.1-cudnn9-runtime-ubuntu22.04 AS base

RUN apt-get update && apt-get install -y python3.10 python3-pip curl && rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY entrypoint.sh /entrypoint.sh
RUN chmod +x /entrypoint.sh

ENV MODEL_PATH=/models MODEL_NAME=llama-3-8b TENSOR_PARALLEL_SIZE=1

HEALTHCHECK --interval=10s --timeout=5s --retries=12 CMD curl -f http://localhost:8000/health

USER nobody
EXPOSE 8000
ENTRYPOINT ["/entrypoint.sh"]
""")

print("entrypoint.sh:")
print("""
#!/bin/bash
set -e
python -m vllm.entrypoints.openai.api_server \\
  --model $MODEL_PATH/$MODEL_NAME \\
  --tensor-parallel-size $TENSOR_PARALLEL_SIZE \\
  --port 8000 &
# Wait for server to be ready
until curl -sf http://localhost:8000/health; do sleep 2; done
echo "Server ready"
wait
""")

## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- A production Dockerfile packages the model runtime, startup script, health checks, and metrics endpoint.
- The HEALTHCHECK is the key prod requirement: Kubernetes will not route traffic until health checks pass, naturally implementing a warm-up barrier..
- Day 51 implementation complete.
